# Lunar crater labeling with ipyleaflet

Select a projected lunar GeoTIFF, click a crater center, expand it into a polygon, and export accepted labels to a **GeoPackage**.
Inspired by the interactive map in [NASA eo-validation / ValidationDashboard.ipynb](https://github.com/nasa-nccs-hpda/eo-validation/blob/main/notebooks/ValidationDashboard.ipynb) and its [ValidationDashboard implementation](https://github.com/nasa-nccs-hpda/eo-validation/blob/main/eo_validation/validation_dashboard.py).

The **ipyleaflet** map supports mouse-wheel zoom, drag-to-pan, fullscreen, layer visibility, and draggable polygon vertices, and automatic saving as you label. Raster detail refreshes automatically after navigation. Image windows are sent through notebook widgets, so no extra tile-server port or JupyterHub proxy configuration is needed.

### Setup
Install `requirements/requirements_labeling.txt` into the notebook kernel environment. After upgrading from the Matplotlib version, restart the kernel and reload the browser page. JupyterLab 4 supports the installed ipyleaflet widget extension. On a managed JupyterHub with a separately managed frontend environment, the administrator may also need to install `jupyter-leaflet` in that frontend environment.

### Lunar coordinates
The map uses Leaflet's **Simple planar CRS** with locations mapped to **native northing/easting**, not Earth latitude/longitude. The raster's projected lunar CRS remains authoritative; image bounds, clicks, polygons, and output all share that coordinate plane. Geographic rasters must first be projected into an appropriate lunar CRS. There is no Earth basemap, Earth reprojection, or Earth geodesic measurement. Ellipse radii use the raster's native linear units (metres for the supplied example); region-growing limits use source pixels. Rotated rasters are displayed using a north-up VRT in the same CRS.

In [ ]:
# Run once if needed, then restart the kernel:
# %pip install -r ../requirements/requirements_labeling.txt

In [ ]:
from pathlib import Path
import sys
from IPython.display import display

"""
repo_root = Path.cwd().parent
repo_root_str = str(repo_root).replace('/panfs/ccds02/nobackup', '/explore/nobackup')
repo_root = Path(repo_root_str)
NOTEBOOK_DIR = repo_root / "notebooks"

if not (repo_root / "lfm").exists():
  raise FileNotFoundError(
      "Cannot find lfm/ directory. Run this notebook from "
      "lfm/notebooks/full_model or update repo_root."
  )
sys.path.insert(0, str(repo_root))

"""
# Works when the kernel starts in the repository root or notebooks/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "lfm" / "labeling" / "craters.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Start this notebook from the lfm repository or set ROOT to its path.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lfm.labeling.craters import CraterLabeler

### Label and save as you go

1. Under **1. Raster to label**, click **Browse raster…** to open the compact `ipyfilechooser` file dialog (the same chooser used by leafmap). Navigate folders, highlight a GeoTIFF, then click **Load raster**. The chooser collapses after loading; its own select button also loads the chosen raster. The configured Explore file is the default and loads automatically when available. On a machine without that filesystem, the chooser starts at the nearest existing parent; browse to a local raster. It browses the **notebook server's filesystem**, not files on a separate laptop when connected to JupyterHub.
2. **Scroll to zoom** and **drag to pan**. Higher-detail imagery refreshes automatically. Use **Navigate** to explore without adding labels.
3. **Edge circle** is the default. Click a crater's geometric center: a valid edge-fit circle is **saved immediately**. Adjust **Min radius / Max radius** and click **Fit circle to edges** if needed. A poor fit can be resized or deleted.
4. For a manual circle, choose **Circle**, click its center, and set its size using either the **radius slider** or the **numeric input**. Both show the same radius in native map units (metres for the example). Releasing the slider or committing a typed value updates the **same crater** and saves it; no Add/Accept button is needed. **Ellipse** also uses axis ratio and angle. Clicking outside existing outlines starts another crater. Clicking an existing crater selects it instead of adding a duplicate; the size controls update that same label. Double-click a crater to refit its circular edge; this keeps Edge circle active. Choose **Edit vertices** explicitly if you want draggable handles.
5. **Edit vertices** refines the selected outline; valid edits save automatically. **Region grow** remains available as an intensity-based proposal, separate from edge-circle fitting.
6. To edit or delete any saved/imported crater, click inside it in any mode except Navigate, or select it from the **Crater** list. Use the radius slider/input to resize the selected outline, or choose **Edit vertices** explicitly. Double-clicking in a drawing mode refits a circular edge without adding another label. Click **Delete selected**; deletion saves immediately. **Undo change** restores a deletion or previous geometry in the current session (up to 30 changes). To redraw, return to a drawing mode and click a new center. The smallest containing crater is selected where outlines overlap; the list gives access to larger overlapping craters.
7. The cyan outline is the selected crater; green outlines are saved craters. The latest save path is displayed below the controls. Every successful click/edit/import/deletion updates `<raster_stem>_label_craters.gpkg` under `LABEL_DIR`. Reopening a raster reloads its saved labels. Invalid outlines and failed writes leave the previous saved version intact, retain the unsaved edit, and report an error. Correct it and save, or use **Undo change** to revert. **Export GeoPackage** retries pending changes or explicitly saves the current collection. Use one session per output file; concurrent writers are not merged.

### Start from an existing crater catalog

Expand **2. Existing crater vectors / catalog (optional)** and choose a vector file or CSV catalog with **Browse catalog…**, then click **Import clipped catalog** after loading the raster. This adds intersecting crater polygons, clips them to the raster's actual footprint (including rotated scenes), and saves them alongside existing labels. Importing the same catalog again skips identifiers already present. Explicitly importing again can restore previously deleted catalog features. Imports are explicit for each scene; changing rasters does not automatically reimport deleted labels.

Supported inputs:

- **Robbins CSV**: `CRATER_ID`, `LAT_CIRC_IMG`, `LON_CIRC_IMG`, `DIAM_CIRC_IMG`.
- **Generic lunar CSV**: `latitude`, `longitude`, `diameter_km`, optional `id`. Latitude is planetocentric; longitude is east-positive (0–360 or −180–180); diameter is kilometres on the 1737.4 km lunar sphere. Catalog centers/diameters become geodesic circular rim proposals before projection and clipping.
- **Polygon GeoPackage, shapefile, or GeoJSON with an explicit lunar CRS**: polygons are reprojected and clipped. A GeoJSON interpreted as Earth WGS84 is rejected; assign the actual lunar CRS in a GeoPackage first. For multi-layer GeoPackages the first layer is read.

The downloaded full [USGS Robbins catalog](https://astrogeology.usgs.gov/search/map/moon_crater_database_v1_robbins) is in `data/catalogs/lunar_crater_database_robbins_2018.csv` on this machine (data/ is git-ignored). A small **downloaded-data subset** is included in `tests/fixtures/robbins_scene.csv`, with provenance next to it: two original rows whose circles intersect the supplied NAC scene. The notebook uses the full catalog when present and the portable subset otherwise. To download the full catalog on Explore, run `python scripts/python/download_crater_catalog.py` from the repository root (~92 MB download, ~238 MB CSV). This global catalog targets craters roughly ≥1–2 km; it will not supply most small NAC craters. Catalog circles are starting labels, not precise traced rims.

### Interpreting automatic outlines

Edge circle fixes your clicked center and searches for radial contrast across candidate radii. It uses bounded image windows and CPU processing without model weights. Shadows, ejecta and neighboring craters can produce the wrong edge; the displayed sector-support fraction is not a calibrated confidence score. Region grow follows intensity and can outline only the shadowed portion. Review and refine saved proposals, or delete and redraw them.

Export layer: `craters`, in the original lunar CRS. Attributes include `crater_id`, `method`, `source`, `band`, `seed_col`, `seed_row`, `area_native` (squared CRS units), `catalog_id`, and `catalog_source`. Existing label files from the earlier notebook remain readable; their catalog metadata is blank.


In [ ]:
# Browse files opens a folder navigator at the default raster directory.
# Use it to navigate the notebook server filesystem and choose a GeoTIFF.
DEFAULT_RASTER = Path("/explore/nobackup/projects/lfm/Benchmarks/Craters/NAC_PHO_E064S3160/NAC_DTM_NEWCRATER6_M1219245090_80CM.TIF")
RASTER_DIR = DEFAULT_RASTER.parent
LABEL_DIR = ROOT / "outputs" / "labels"

# Accepted labels are reloaded from the raster-specific GeoPackage on Load raster.
# Accept or discard the current draft before rerunning this cell.
if "dashboard" in globals():
    dashboard.close()
dashboard = CraterLabeler(data_dir=RASTER_DIR, output_dir=LABEL_DIR,
                          default_raster=DEFAULT_RASTER)
display(dashboard.widget)